In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import cv2

#Definimos el algoritmo de clústering por k-means
def segment_gmm_intensity(img, K=3):
    h, w = img.shape

    X = img.reshape(-1, 1)  # solo intensidad

    gmm = GaussianMixture(n_components=K, covariance_type='full', random_state=0)
    gmm.fit(X)

    labels = gmm.predict(X).reshape(h, w)

    # Elegir el clúster más brillante
    means = gmm.means_.flatten()
    cluster_tumor = np.argmax(means)

    mask = (labels == cluster_tumor).astype(np.uint8)

    return mask, labels, means

def opening_by_reconstruction(mask, iterations=1):
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))

    # erosión
    eroded = cv2.erode(mask, ker, iterations=iterations)

    # reconstrucción morfológica
    reconstructed = cv2.dilate(eroded, ker, iterations=iterations)
    reconstructed = np.minimum(reconstructed, mask)

    return reconstructed



def compute_roundness(area, perimeter):
    if perimeter == 0:
        return 0
    return (4 * np.pi * area) / (perimeter ** 2)


def clean_mask(mask, k = 1, alpha = 0.5, area_percent = 0.01):
    # 1) Apertura morfológica para quitar puntos sueltos
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # 2) Cierre para rellenar huecos internos
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)

    # ⭐ 2.5) Opening by reconstruction para romper cuellos finos
    mask_clean = opening_by_reconstruction(mask_clean, iterations=1)

    # 3) Componentes conectadas
    num_labels, labels = cv2.connectedComponents(mask_clean)

    if num_labels <= 1:
        return mask_clean  # no hay regiones

    areas = []
    roundnesses = []

    # Calcular métricas por componente
    for lab in range(1, num_labels):
        comp = (labels == lab).astype(np.uint8)

        # Área
        area = comp.sum()

        # Perímetro
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(cnts) == 0:
            perimeter = 0
        else:
            perimeter = cv2.arcLength(cnts[0], True)

        roundness = compute_roundness(area, perimeter)

        areas.append(area)
        roundnesses.append(roundness)

    areas = np.array(areas, dtype=float)
    roundnesses = np.array(roundnesses, dtype=float)

    # -----------------------------
    # 🔥 ELIMINAR pegotes pequeños
    # -----------------------------
    total_area = (mask > 0).sum()
    min_area = area_percent * total_area
    
    valid = areas >= min_area
    if not np.any(valid):
        return np.zeros_like(mask, dtype=np.uint8)

    # Filtrar arrays
    areas_f = areas[valid]
    round_f = roundnesses[valid]

    # Guardar qué etiquetas sobreviven
    surviving_labels = np.where(valid)[0] + 1
    # -----------------------------

    # Normalizar área entre 0 y 1 (solo válidos)
    norm_area = (areas_f - areas_f.min()) / (areas_f.max() - areas_f.min() + 1e-8)

    # Score combinado
    score = alpha * norm_area + (1 - alpha) * round_f

    # Elegir los k mejores entre los válidos
    idx_top = np.argsort(score)[::-1][:k]
    labels_top = surviving_labels[idx_top]

    # Construir máscara final
    final_mask = np.isin(labels, labels_top).astype(np.uint8)

    return final_mask

In [ ]:
#Lo aplicamos:
#Probamos para una imagen, y comparamos con las máscaras binarias


carpeta_origen = 'Imagenes png INbreast preprocesadas RENOMBRADAS'
carpeta_binarias = 'Imagenes mascaras segmentacion INbreast preprocesadas RENOMBRADAS'
files = os.listdir(carpeta_origen)

#PROBAMOS SOLO PARA LOS QUE PONEN MASS (OPCIONAL)
files = [x for x in files if x.startswith("MASS")]

nombre_imagen = files[30]    #<---- esto hay q cambiar

nombre_mascara = nombre_imagen.replace('.png','_mask.png')
ruta_imagen = os.path.join(carpeta_origen, nombre_imagen)
ruta_mascara = os.path.join(carpeta_binarias, nombre_mascara)

#Importamos la foto
im = Image.open(ruta_imagen)
im = np.array(im)

#Hacemos los k-means
mascara, imagen_clusters, medias = segment_gmm_intensity(im, K = 4)

#Y pintamos la imagen normal
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(im, cmap="gray")
axes[0].axis("off")
axes[0].set_title('Imagen normal')

#Ploteamos todo, a ver:

axes[1].imshow(mascara, cmap="gray")
axes[1].axis("off")
axes[1].set_title('Máscara creada por K-means')


#Aplicamos y pintamos ahora la máscara filtrada con nuestro método nuevo:
mascara_limpia = clean_mask(mascara, k = 5, alpha = 0.2, area_percent = 0.01)

axes[2].imshow(mascara_limpia, cmap="gray")
axes[2].axis("off")
axes[2].set_title('Máscara K-means filtrada')


#Y comparamos con la máscara verdadera:
masc = Image.open(ruta_mascara)
axes[3].imshow(masc, cmap="gray")
axes[3].axis("off")
axes[3].set_title('Máscara verdadera')

plt.tight_layout()
plt.show()